# Day 3: retry the difficult sections

**How to run:** Runtime → Run all. It takes about 10–15 minutes. When it finishes, your browser downloads `sections_geocoded_retry_raw.csv`. Send that file back to Claude.

This run retries only the 58 "check first" sections. It differs from Day 2 in three ways: it searches only inside greater Hyderabad, it tries more spellings, and when several places share a name it prefers the one nearest the section's already-placed neighbours.

In [ ]:
# ==== CELL 1: the 58 sections to retry, each with an "expected area" ====
# expected area = the middle point of well-placed neighbouring sections
# (same subdivision if possible, else same division, else same circle)
import json, time, re, math
import pandas as pd, requests
TARGETS = json.loads('[{"section_id": "BANJARA_HILLS_SRI_KRISHNA_NAGAR", "section": "SRI KRISHNA NAGAR", "division": "Banjara Hills", "anchor_lat": 17.43683, "anchor_lon": 78.43292, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "CYBERCITY_BALAJI_NAGAR", "section": "BALAJI NAGAR", "division": "Kondapur", "anchor_lat": 17.45658, "anchor_lon": 78.37306, "anchor_basis": "division (3 sections)"}, {"section_id": "CYBERCITY_CHEVELLA", "section": "CHEVELLA", "division": "Ibrahimbagh", "anchor_lat": 17.45879, "anchor_lon": 78.36589, "anchor_basis": "circle (9 sections)"}, {"section_id": "CYBERCITY_CHILKUR", "section": "CHILKUR", "division": "Ibrahimbagh", "anchor_lat": 17.45879, "anchor_lon": 78.36589, "anchor_basis": "circle (9 sections)"}, {"section_id": "CYBERCITY_IBRAHIMBAGH", "section": "IBRAHIMBAGH", "division": "Ibrahimbagh", "anchor_lat": 17.45879, "anchor_lon": 78.36589, "anchor_basis": "circle (9 sections)"}, {"section_id": "CYBERCITY_K_P_H_B_COLONY", "section": "K P H B COLONY", "division": "Kondapur", "anchor_lat": 17.45658, "anchor_lon": 78.37306, "anchor_basis": "division (3 sections)"}, {"section_id": "CYBERCITY_NARSINGI", "section": "NARSINGI", "division": "Ibrahimbagh", "anchor_lat": 17.45879, "anchor_lon": 78.36589, "anchor_basis": "circle (9 sections)"}, {"section_id": "CYBERCITY_SHANKAR_PALLY", "section": "SHANKAR PALLY", "division": "Ibrahimbagh", "anchor_lat": 17.45879, "anchor_lon": 78.36589, "anchor_basis": "circle (9 sections)"}, {"section_id": "HABSIGUDA_GHATKESAR", "section": "GHATKESAR", "division": "Keesara", "anchor_lat": 17.4159, "anchor_lon": 78.60227, "anchor_basis": "subdivision (3 sections)"}, {"section_id": "HABSIGUDA_UPPAL_BAGAYATH", "section": "UPPAL BAGAYATH", "division": "Habsiguda", "anchor_lat": 17.40067, "anchor_lon": 78.54921, "anchor_basis": "subdivision (4 sections)"}, {"section_id": "HYDERABAD_CENTRAL_AMBERPET", "section": "AMBERPET", "division": "Azamabad", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_GOLCONDA", "section": "GOLCONDA", "division": "Mehdipatnam", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_LANGER_HOUSE", "section": "LANGER HOUSE", "division": "Mehdipatnam", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_RETHI_BOWLI", "section": "RETHI BOWLI", "division": "Mehdipatnam", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_SHAIKPET", "section": "SHAIKPET", "division": "Mehdipatnam", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_SHANKARMUTT", "section": "SHANKARMUTT", "division": "Azamabad", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_SURYANAGAR", "section": "SURYANAGAR", "division": "Mehdipatnam", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_CENTRAL_VIJAYA_NAGAR_COLONY", "section": "VIJAYA NAGAR COLONY", "division": "Mehdipatnam", "anchor_lat": 17.40397, "anchor_lon": 78.47718, "anchor_basis": "circle (7 sections)"}, {"section_id": "HYDERABAD_SOUTH_CHANDRAYANA_GUTTA", "section": "CHANDRAYANA GUTTA", "division": "Charminar", "anchor_lat": 17.33734, "anchor_lon": 78.47648, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "HYDERABAD_SOUTH_DATTATREYA_COLONY", "section": "DATTATREYA COLONY", "division": "Begum Bazar", "anchor_lat": 17.37601, "anchor_lon": 78.46478, "anchor_basis": "division (3 sections)"}, {"section_id": "HYDERABAD_SOUTH_MOGHAL_PURA", "section": "MOGHAL PURA", "division": "Charminar", "anchor_lat": 17.36083, "anchor_lon": 78.46675, "anchor_basis": "division (5 sections)"}, {"section_id": "HYDERABAD_SOUTH_PUTLI_BOWLI", "section": "PUTLI BOWLI", "division": "Begum Bazar", "anchor_lat": 17.37601, "anchor_lon": 78.46478, "anchor_basis": "division (3 sections)"}, {"section_id": "HYDERABAD_SOUTH_SAIDABAD", "section": "SAIDABAD", "division": "Asmangadh", "anchor_lat": 17.35705, "anchor_lon": 78.50239, "anchor_basis": "division (2 sections)"}, {"section_id": "MEDCHAL_ALIABAD", "section": "ALIABAD", "division": "Medchal", "anchor_lat": 17.52718, "anchor_lon": 78.48027, "anchor_basis": "division (5 sections)"}, {"section_id": "MEDCHAL_BHAGYA_NAGAR", "section": "BHAGYA NAGAR", "division": "Kukatpally", "anchor_lat": 17.49872, "anchor_lon": 78.41021, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "MEDCHAL_JAGADGIRI_GUTTA", "section": "JAGADGIRI GUTTA", "division": "Medchal", "anchor_lat": 17.52055, "anchor_lon": 78.47066, "anchor_basis": "subdivision (4 sections)"}, {"section_id": "MEDCHAL_MEDCHAL_RURAL", "section": "MEDCHAL RURAL", "division": "Medchal", "anchor_lat": 17.52718, "anchor_lon": 78.48027, "anchor_basis": "division (5 sections)"}, {"section_id": "MEDCHAL_MEDCHAL_TOWN", "section": "MEDCHAL TOWN", "division": "Medchal", "anchor_lat": 17.52718, "anchor_lon": 78.48027, "anchor_basis": "division (5 sections)"}, {"section_id": "MEDCHAL_SHAMEERPET", "section": "SHAMEERPET", "division": "Medchal", "anchor_lat": 17.52718, "anchor_lon": 78.48027, "anchor_basis": "division (5 sections)"}, {"section_id": "MEDCHAL_SHAPURNAGAR", "section": "SHAPURNAGAR", "division": "Jeedimetla", "anchor_lat": 17.53356, "anchor_lon": 78.42483, "anchor_basis": "division (4 sections)"}, {"section_id": "RAJENDRA_NAGAR_BALAPUR", "section": "BALAPUR", "division": "Kandukur", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "circle (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_GAGANPAHAD", "section": "GAGANPAHAD", "division": "Rajendra Nagar", "anchor_lat": 17.31772, "anchor_lon": 78.44503, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "RAJENDRA_NAGAR_HIMAYAT_SAGAR", "section": "HIMAYAT SAGAR", "division": "Rajendra Nagar", "anchor_lat": 17.33162, "anchor_lon": 78.43657, "anchor_basis": "subdivision (3 sections)"}, {"section_id": "RAJENDRA_NAGAR_KOTHUR", "section": "KOTHUR", "division": "Shadnagar", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "circle (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_MAHESHWARAM", "section": "MAHESHWARAM", "division": "Kandukur", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "circle (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_MD_PALLY", "section": "MD PALLY", "division": "Rajendra Nagar", "anchor_lat": 17.31772, "anchor_lon": 78.44503, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "RAJENDRA_NAGAR_NANDIGAMA", "section": "NANDIGAMA", "division": "Shadnagar", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "circle (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_PAHADISHARIFF", "section": "PAHADISHARIFF", "division": "Kandukur", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "circle (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_SHADNAGAR_RURAL", "section": "SHADNAGAR RURAL", "division": "Shadnagar", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "circle (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_SHAMSHABAD", "section": "SHAMSHABAD", "division": "Rajendra Nagar", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "division (5 sections)"}, {"section_id": "RAJENDRA_NAGAR_SHAPOOR", "section": "SHAPOOR", "division": "Rajendra Nagar", "anchor_lat": 17.33162, "anchor_lon": 78.43982, "anchor_basis": "division (5 sections)"}, {"section_id": "SAROORNAGAR_ABDULLAPURMET", "section": "ABDULLAPURMET", "division": "Ibrahimpatnam", "anchor_lat": 17.34337, "anchor_lon": 78.54685, "anchor_basis": "circle (8 sections)"}, {"section_id": "SAROORNAGAR_ADIBATLA", "section": "ADIBATLA", "division": "Ibrahimpatnam", "anchor_lat": 17.34337, "anchor_lon": 78.54685, "anchor_basis": "circle (8 sections)"}, {"section_id": "SAROORNAGAR_BN_REDDY_NAGAR", "section": "BN REDDY NAGAR", "division": "Champapet", "anchor_lat": 17.34216, "anchor_lon": 78.52221, "anchor_basis": "division (3 sections)"}, {"section_id": "SAROORNAGAR_INJAPUR", "section": "INJAPUR", "division": "Ibrahimpatnam", "anchor_lat": 17.34337, "anchor_lon": 78.54685, "anchor_basis": "circle (8 sections)"}, {"section_id": "SAROORNAGAR_LENIN_NAGAR", "section": "LENIN NAGAR", "division": "Champapet", "anchor_lat": 17.34146, "anchor_lon": 78.51959, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "SAROORNAGAR_MEERPET", "section": "MEERPET", "division": "Champapet", "anchor_lat": 17.34146, "anchor_lon": 78.51959, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "SAROORNAGAR_PEDDA_AMBERPET", "section": "PEDDA AMBERPET", "division": "Saroornagar", "anchor_lat": 17.33315, "anchor_lon": 78.5776, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "SAROORNAGAR_RAMAKRISHNAPURAM", "section": "RAMAKRISHNAPURAM", "division": "Saroornagar", "anchor_lat": 17.34985, "anchor_lon": 78.57273, "anchor_basis": "division (5 sections)"}, {"section_id": "SAROORNAGAR_TURKAYAMZAL", "section": "TURKAYAMZAL", "division": "Ibrahimpatnam", "anchor_lat": 17.34337, "anchor_lon": 78.54685, "anchor_basis": "circle (8 sections)"}, {"section_id": "SAROORNAGAR_VANASTHALIPURAM", "section": "VANASTHALIPURAM", "division": "Champapet", "anchor_lat": 17.34216, "anchor_lon": 78.52221, "anchor_basis": "division (3 sections)"}, {"section_id": "SECUNDERABAD_BALANAGAR", "section": "BALANAGAR", "division": "Bowenpally", "anchor_lat": 17.44679, "anchor_lon": 78.50886, "anchor_basis": "circle (11 sections)"}, {"section_id": "SECUNDERABAD_FEROZEGUDA", "section": "FEROZEGUDA", "division": "Bowenpally", "anchor_lat": 17.44679, "anchor_lon": 78.50886, "anchor_basis": "circle (11 sections)"}, {"section_id": "SECUNDERABAD_I_D_P_L", "section": "I.D.P.L.", "division": "Bowenpally", "anchor_lat": 17.44679, "anchor_lon": 78.50886, "anchor_basis": "circle (11 sections)"}, {"section_id": "SECUNDERABAD_LALBAZAR", "section": "LALBAZAR", "division": "Secunderabad", "anchor_lat": 17.50528, "anchor_lon": 78.51626, "anchor_basis": "subdivision (3 sections)"}, {"section_id": "SECUNDERABAD_MARREDPALLY", "section": "MARREDPALLY", "division": "Secunderabad", "anchor_lat": 17.46, "anchor_lon": 78.50057, "anchor_basis": "subdivision (2 sections)"}, {"section_id": "SECUNDERABAD_OLD_BOWENPALLY", "section": "OLD BOWENPALLY", "division": "Bowenpally", "anchor_lat": 17.44679, "anchor_lon": 78.50886, "anchor_basis": "circle (11 sections)"}, {"section_id": "SECUNDERABAD_RANGA_REDDY_NAGAR", "section": "RANGA REDDY NAGAR", "division": "Bowenpally", "anchor_lat": 17.44679, "anchor_lon": 78.50886, "anchor_basis": "circle (11 sections)"}]')
print(len(TARGETS), "sections to retry")

In [ ]:
# ==== CELL 2: extra spellings to try ====
EXTRA = {
    "SHANKAR PALLY": ["Shankarpalle", "Shankarpally", "Shankarpalli"],
    "UPPAL BAGAYATH": ["Uppal Bhagayath", "Uppal Bhagath", "Uppal Bagayat"],
    "RETHI BOWLI": ["Rethibowli", "Rethi Bowli", "Rethibowli Mehdipatnam"],
    "SHANKARMUTT": ["Shankar Mutt", "Shankermutt", "Shankar Math"],
    "CHANDRAYANA GUTTA": ["Chandrayangutta", "Chandrayanagutta"],
    "MOGHAL PURA": ["Moghalpura", "Mughalpura"],
    "PUTLI BOWLI": ["Putlibowli", "Putli Bowli"],
    "JAGADGIRI GUTTA": ["Jagadgirigutta", "Jagathgirigutta"],
    "SHAPURNAGAR": ["Shapur Nagar", "Shapurnagar"],
    "TURKAYAMZAL": ["Turkayamjal", "Turkayamzal"],
    "FEROZEGUDA": ["Ferozguda", "Feroz Guda"],
    "LALBAZAR": ["Lal Bazar", "Lal Bazaar"],
    "MD PALLY": ["Mailardevpally", "Mylardevpally", "Mailardevpalli"],
    "K P H B COLONY": ["KPHB Colony", "Kukatpally Housing Board Colony"],
    "I.D.P.L.": ["IDPL Colony", "IDPL Township", "IDPL"],
    "MEDCHAL RURAL": ["Medchal"], "MEDCHAL TOWN": ["Medchal"],
    "SHADNAGAR RURAL": ["Shadnagar"],
    "PAHADISHARIFF": ["Pahadi Shareef", "Pahadishareef"],
    "RANGA REDDY NAGAR": ["Ranga Reddy Nagar", "RR Nagar"],
    "BN REDDY NAGAR": ["BN Reddy Nagar", "B N Reddy Nagar"],
    "VIJAYA NAGAR COLONY": ["Vijayanagar Colony", "Vijay Nagar Colony"],
    "SRI KRISHNA NAGAR": ["Sri Krishna Nagar", "Krishna Nagar"],
    "DATTATREYA COLONY": ["Dattatreya Nagar", "Dattatreya Colony"],
    "RAMAKRISHNAPURAM": ["Ramakrishnapuram", "RK Puram"],
    "HIMAYAT SAGAR": ["Himayatsagar"],
    "OLD BOWENPALLY": ["Old Bowenpally", "Bowenpally"],
}
def variants(section):
    v = [section.title()]
    if " " in section:
        v.append(section.replace(" ", "").title())          # e.g. Moghalpura
    v += EXTRA.get(section, [])
    return list(dict.fromkeys(v))
def norm(s): return re.sub(r"[^a-z]", "", str(s).lower())

In [ ]:
# ==== CELL 3: search again, only inside greater Hyderabad (about 10-15 minutes) ====
HEADERS = {"User-Agent": "EV-charger-siting research project (github.com/unnatisingh12/EV-charger-siting)"}
API = "https://nominatim.openstreetmap.org/search"
VIEWBOX = "77.9,17.9,79.1,16.8"   # results OUTSIDE this box are not returned this time
AREA = {"place", "boundary"}

def km(a, b, c, d):
    a, b, c, d = map(math.radians, [a, b, c, d])
    h = math.sin((c-a)/2)**2 + math.cos(a)*math.cos(c)*math.sin((d-b)/2)**2
    return 6371*2*math.asin(math.sqrt(h))

def score(cand, t, v):
    """Lower is better: distance from the expected area, plus penalties."""
    d = km(t["anchor_lat"], t["anchor_lon"], float(cand["lat"]), float(cand["lon"]))
    pen = 0
    if cand.get("category") not in AREA: pen += 3            # a shop/road, not the area
    if (cand.get("place_rank") or 30) <= 16: pen += 10       # whole mandal / city
    first = cand.get("display_name", "").split(",")[0]
    if norm(v)[:6] not in norm(first) and norm(first)[:6] not in norm(v): pen += 5   # name doesn't match
    return round(d + pen, 2), round(d, 2)

rows = []
for i, t in enumerate(TARGETS):
    cands, stop = [], False
    for v in variants(t["section"]):
        for q in [f"{v}, Hyderabad", f"{v}, {t['division']}", v]:
            try:
                r = requests.get(API, headers=HEADERS, timeout=30, params={
                    "q": q, "format": "jsonv2", "limit": 10, "countrycodes": "in",
                    "viewbox": VIEWBOX, "bounded": 1})
                res = r.json() if r.status_code == 200 else []
            except Exception:
                res = []
            time.sleep(1.1)
            for c in res:
                s, d = score(c, t, v)
                cands.append({"score": s, "km_from_expected": d, "query": q,
                              "lat": float(c["lat"]), "lon": float(c["lon"]),
                              "name": c.get("display_name"), "category": c.get("category"),
                              "type": c.get("type"), "place_rank": c.get("place_rank")})
            best = min(cands, key=lambda x: x["score"]) if cands else None
            limit = 8 if t["anchor_basis"].startswith("circle") else 4
            if best and best["score"] <= limit:
                stop = True; break
        if stop: break
    seen, uniq = set(), []
    for c in sorted(cands, key=lambda x: x["score"]):
        k = (round(c["lat"], 4), round(c["lon"], 4))
        if k not in seen: seen.add(k); uniq.append(c)
    b = uniq[0] if uniq else {}
    rows.append({"section_id": t["section_id"], "section": t["section"],
                 "retry_status": "found" if uniq else "not_found",
                 "anchor_lat": t["anchor_lat"], "anchor_lon": t["anchor_lon"], "anchor_basis": t["anchor_basis"],
                 **{f"best_{k}": v for k, v in b.items()},
                 "other_candidates": json.dumps(uniq[1:5])})
    print(f"{i+1:>2}/{len(TARGETS)} {t['section']:<22} -> " +
          (f"{b['name'][:55]}  ({b['km_from_expected']} km from expected)" if uniq else "not found"))

out = pd.DataFrame(rows)
print("\nFound:", (out.retry_status == "found").sum(), " Not found:", (out.retry_status == "not_found").sum())

In [ ]:
# ==== CELL 4: save and download ====
out.to_csv("sections_geocoded_retry_raw.csv", index=False)
try:
    from google.colab import files
    files.download("sections_geocoded_retry_raw.csv")
except ImportError:
    print("Saved sections_geocoded_retry_raw.csv")